# 第16周 · Flow Matching：把噪声搬运成数据（Capstone 基线检查）

**本周结束时你能做到**：把生成建模理解为"把一团噪声搬运成数据"；把常微分方程（ODE）读成"粒子在速度场里流动"，并会用欧拉法模拟；写出 Flow Matching 的线性路径、条件速度目标与损失，认识到它的训练**就是一个回归问题**；在 2D 分布上训练、采样、画轨迹，并用两个互补的指标评估样本；认出"容量不足"和"步数不足"的典型失败形态。

本周的蓝本：

- **MIT 6.S184（2026）讲义**：Peter Holderrieth & Ezra Erives, *An Introduction to Flow Matching and Diffusion Models*（[arXiv:2506.02070](https://arxiv.org/abs/2506.02070)，v3；课程网站 [diffusion.csail.mit.edu](https://diffusion.csail.mit.edu/)，网站内容 CC BY-NC-SA）——欧拉采样照 §2.1 **Algorithm 1**，Flow Matching 训练照 §3.3 **Algorithm 3**。下文简称"MIT 讲义"。
- **MIT 课程的 lab 仓库** [eje24/iap-diffusion-labs](https://github.com/eje24/iap-diffusion-labs)（MIT License）：训练循环的结构改编自 Lab 2；作业里用的小型 U-Net 改编自 `2025` 分支 Lab 3 的 `MNISTUNet`，放在课程工具包里（`generative.SmallUNet`）。
- 数学只保留到两个直观层面——"粒子沿速度场流动"和"高斯插值"；不讲随机微分方程（SDE）与 Fokker–Planck 方程。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from zero2researcher import data, experiments, generative
from zero2researcher.checks import week16 as ck

# 本节全部是 2D 小实验：固定用 CPU（这么小的网络在 GPU 上反而更慢，也保证数值与讲义一致）
device = torch.device("cpu")
print("torch", torch.__version__, "| device:", device)

---
## S1 · 从噪声到数据：粒子在速度场里流动

生成模型要解决的问题：手里有一堆样本（手写数字、照片、蛋白质结构……），它们来自某个我们写不出公式的分布 $p_{\text{data}}$；我们想**造出新的、看起来同样真实的样本**。

本周的思路（MIT 讲义第 1–2 章）分三步：

1. 从一个**容易采样**的分布出发——标准正态噪声 $x_0 \sim \mathcal{N}(0, I)$；
2. 把每个噪声点看成一颗**粒子**，让它沿着一个**速度场** $u(x, t)$ 从时刻 $t = 0$ 走到 $t = 1$；
3. 如果速度场选得好，$t = 1$ 时这群粒子的分布恰好就是 $p_{\text{data}}$。

"粒子沿速度场流动"写成数学，就是常微分方程（ODE）

$$\frac{dx}{dt} = u(x, t)$$

它只说了一件事：**粒子在位置 $x$、时刻 $t$ 的速度是 $u(x, t)$**。

### 欧拉法：每次沿当前速度走一小步

速度场已知时，最朴素的模拟办法是把 $[0, 1]$ 切成 $n$ 段、步长 $h = 1/n$，每一步按**当前位置、当前时刻**的速度直走一小段（MIT 讲义 §2.1 Algorithm 1）：

$$x_{t+h} = x_t + h \cdot u(x_t, t)$$

先拿一个**知道答案**的速度场练手：绕原点逆时针旋转的场 $u(x, y) = (-y,\ x)$。粒子走 $\Delta t$ 的时间就转过 $\Delta t$ 弧度——从 $t = 0$ 走到 $t = \pi/2$，应该恰好转 90°，$(x, y) \to (-y, x)$。

In [ ]:
def euler_integrate(u, x0, n_steps, t_start=0.0, t_end=1.0):
    """用欧拉法从 t_start 积分到 t_end。返回整条轨迹，形状 (n_steps + 1, *x0.shape)，第 0 行是起点"""
    h = (t_end - t_start) / n_steps
    x = x0
    path = [x0]
    for i in range(n_steps):
        t = t_start + i * h          # 这一步起点的时刻
        x = x + h * u(x, t)          # 沿当前速度走一小步
        path.append(x)
    return torch.stack(path)


def rotation(x, t):
    # 逆时针旋转场 u(x, y) = (−y, x)，与时间无关
    return torch.stack([-x[:, 1], x[:, 0]], dim=1)


torch.manual_seed(0)
particles = torch.randn(300, 2) * 0.3 + torch.tensor([2.0, 0.0])     # 一团中心在 (2, 0) 的粒子
exact_end = torch.stack([-particles[:, 1], particles[:, 0]], dim=1)   # 精确解：每颗粒子恰好转 90°
for n in (5, 20, 200):
    end = euler_integrate(rotation, particles, n, 0.0, np.pi / 2)[-1]
    err = (end - exact_end).norm(dim=1).mean().item()
    print(f"欧拉 {n:3d} 步：终点与精确解的平均距离 {err:.4f}")
ck.euler_integrate(euler_integrate)

In [ ]:
theta = np.linspace(0, np.pi / 2, 100)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.6))
for ax, n in zip(axes, (5, 200)):
    path = euler_integrate(rotation, particles, n, 0.0, np.pi / 2)
    for k in range(30):                                   # 只画 30 颗粒子的轨迹
        ax.plot(path[:, k, 0], path[:, k, 1], "-", color="C0", lw=0.8, alpha=0.6)
    ax.scatter(particles[:, 0], particles[:, 1], s=4, color="gray", label="Start (t = 0)")
    ax.scatter(path[-1, :, 0], path[-1, :, 1], s=4, color="C3", label="Euler end point")
    ax.plot(2 * np.cos(theta), 2 * np.sin(theta), "k--", lw=1, label="Exact path of the centre")
    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(f"{n} Euler steps")
    ax.legend(fontsize=8, loc="lower left")
fig.suptitle("With few Euler steps particles drift outward; with many they follow the exact rotation")
plt.tight_layout()
plt.show()

欧拉法每一步都沿着**切线**直走，而旋转场的真实轨迹是圆弧——所以每一步都往外偏一点，步数少时偏得明显。步数越多越接近精确解：误差大致与步长 $h$ 成正比（练习 E1 会量出来）。记住这个权衡：**之后生成样本时，"采样步数"就是欧拉步数**——步数少则快但不准，作业 ① 会定量研究它。

### 如果终点已知

再看一个知道答案的场：让所有粒子沿直线、**恰好在 $t = 1$** 到达同一个点 $c$。速度就是"剩下的路程 ÷ 剩下的时间"：

$$u(x, t) = \frac{c - x}{1 - t}$$

In [ ]:
c = torch.tensor([2.0, 1.0])


def to_point(x, t):
    # 剩下的路程 ÷ 剩下的时间：每颗粒子沿直线前进，恰好在 t = 1 到达 c
    return (c - x) / (1 - t)


torch.manual_seed(0)
noise = torch.randn(300, 2)
path = euler_integrate(to_point, noise, 20)      # 欧拉法只在 t = 0, 0.05, …, 0.95 求速度，不会碰到 t = 1 的除零
print("t = 1 时所有粒子与 c 的最大距离:", (path[-1] - c).norm(dim=1).max().item())

fig, ax = plt.subplots(figsize=(4.6, 4.2))
for k in range(60):
    ax.plot(path[:, k, 0], path[:, k, 1], color="C0", lw=0.6, alpha=0.5)
ax.scatter(noise[:, 0], noise[:, 1], s=4, color="gray", label="Noise (t = 0)")
ax.scatter([2.0], [1.0], s=60, color="C3", marker="*", label="Target point c")
ax.set_aspect("equal")
ax.set_title("A known destination gives a known velocity field")
ax.legend(fontsize=8)
plt.show()

数据只有一个点时，速度场一眼就能写出来。**真正的难题是：数据有成千上万个点，哪颗粒子该去哪里？** 能把噪声云搬成数据云的速度场有无穷多个，而我们一个公式都写不出来。

Flow Matching 的回答出奇地简单：**让神经网络从例子里学出速度场**——而且训练只需要一个均方误差损失。

---
## S2 · Flow Matching：把"学速度场"变成一个回归问题

本节的公式与训练流程照 MIT 讲义 §3.3 **Algorithm 3**；训练循环的结构改编自 MIT Lab 2（MIT License），去掉了原 lab 的抽象类层次，只保留"抽时间 → 插值 → MSE"三行。同一个想法在 2022 年秋被几组人几乎同时提出：Flow Matching（Lipman et al., [arXiv:2210.02747](https://arxiv.org/abs/2210.02747)）、Rectified Flow（Liu, Gong & Liu, [arXiv:2209.03003](https://arxiv.org/abs/2209.03003)）与 Stochastic Interpolants（Albergo & Vanden-Eijnden, [arXiv:2209.15571](https://arxiv.org/abs/2209.15571)）。

### 一条直线、一个速度

随便抽一个噪声点 $x_0 \sim \mathcal{N}(0, I)$ 和一个数据点 $x_1 \sim p_{\text{data}}$，用直线把它们连起来：

$$x_t = (1 - t)\,x_0 + t\,x_1, \qquad t \in [0, 1]$$

沿这条直线匀速走，速度处处是 $\dfrac{dx_t}{dt} = x_1 - x_0$，与 $t$ 无关。固定 $x_1$ 和 $t$ 时，$x_t$ 是均值 $t\,x_1$、标准差 $1 - t$ 的高斯——所以这叫**高斯插值**（MIT 讲义称它为 CondOT 概率路径）。

> ⚠️ **时间约定**：本课程沿用 MIT 讲义——**$t = 0$ 是噪声，$t = 1$ 是数据**。下周的 DDPM 论文用的是相反的方向（$t = 0$ 是数据），MIT 讲义附录 E（Summary 45）专门列出了文献里这种反转的约定（原文："Inverted time convention: t=0 corresponds to p_data"）。读任何一篇扩散/流模型的论文，第一件事是确认 $t = 0$ 指的是什么。

### 训练：回归每条直线的速度

我们不知道"整团噪声应该怎样流向数据"，但对**每一对** $(x_0, x_1)$，直线的速度 $x_1 - x_0$ 是现成的。Flow Matching 的训练就是：

1. 抽数据 $x_1$、噪声 $x_0$、时间 $t \sim U(0, 1)$；
2. 算出直线上的位置 $x_t$；
3. 让网络 $u_\theta(x_t, t)$ 预测这条直线的速度，损失是均方误差

$$\mathcal{L}(\theta) = \mathbb{E}_{t,\,x_0,\,x_1}\big\| u_\theta(x_t, t) - (x_1 - x_0) \big\|^2$$

**这和第 4 周的线性回归是同一种问题**：输入 $(x_t, t)$，标签 $x_1 - x_0$，最小化 MSE。区别只有两点：模型换成了 MLP；训练集"无穷大"——每一步都重新抽 $t$ 和噪声，同一个数据点每次都配上新的噪声。

**为什么学到的速度场能把噪声变成数据？** 许多条直线会在同一时刻穿过同一个位置 $x$，它们的速度各不相同；MSE 回归学到的是它们的**平均速度**。Lipman 等人证明了：沿这个"平均速度场"流动，整团粒子在每个时刻的分布恰好就是所有直线上的点在该时刻的分布——到 $t = 1$ 就是数据分布。我们不推导，用实验验证。

先看两份数据：月牙（两个交错的半月）和双高斯。

In [ ]:
DATA = {
    "moons": lambda n: generative.sample_moons(n),
    "gaussian mixture": lambda n: generative.sample_gaussian_mixture(n),     # 中心 (±3, 0)，标准差 0.7
}
torch.manual_seed(0)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
noise_demo = torch.randn(2000, 2)
axes[0].scatter(noise_demo[:, 0], noise_demo[:, 1], s=2, color="gray")
axes[0].set_title("Noise N(0, I): where t = 0 starts")
for ax, (name, sample) in zip(axes[1:], DATA.items()):
    x = sample(2000)
    ax.scatter(x[:, 0], x[:, 1], s=2)
    ax.set_title(f"Data ({name}): where t = 1 should end")
for ax in axes:
    ax.set_xlim(-6, 6)
    ax.set_ylim(-4, 4)
    ax.set_aspect("equal")
fig.suptitle("Generative modelling as transport: from easy noise to the data")
plt.tight_layout()
plt.show()

写出插值与目标，再看看"所有直线上的点"在不同时刻长什么样——这正是训练时网络看到的输入：

In [ ]:
def interpolant(x0, x1, t):
    # 噪声 x0 与数据 x1 之间的直线；t 的形状 (B, 1)，会自动广播到 (B, 2)
    return (1 - t) * x0 + t * x1


def fm_target(x0, x1):
    # 沿直线匀速前进的速度，与 t 无关
    return x1 - x0


ck.interpolant(interpolant)
ck.fm_target(fm_target)

torch.manual_seed(0)
x1 = DATA["moons"](2000)
x0 = torch.randn_like(x1)               # 每个数据点随机配一个噪声点
fig, axes = plt.subplots(1, 5, figsize=(15, 3.2))
for ax, tv in zip(axes, (0.0, 0.25, 0.5, 0.75, 1.0)):
    xt = interpolant(x0, x1, torch.full((len(x1), 1), tv))
    ax.scatter(xt[:, 0], xt[:, 1], s=1)
    ax.set_title(f"t = {tv}")
    ax.set_xlim(-6, 6)
    ax.set_ylim(-4, 4)
    ax.set_aspect("equal")
fig.suptitle("Points on all the straight lines: noise at t = 0 slides into the moons at t = 1")
plt.tight_layout()
plt.show()

网络要学的是：在任意一帧、任意位置，**粒子平均往哪个方向、以多快的速度走**。

速度场网络 $u_\theta(x, t)$ 就是一个 MLP，唯一的新东西是**时间也是输入**：把 $t$ 拼在坐标后面，输入 3 维、输出 2 维（教案配置 3→128→128→2）。激活函数用 SiLU（$x \cdot \mathrm{sigmoid}(x)$，MIT lab 的选择，它让速度场处处光滑）。

In [ ]:
class VelocityMLP(nn.Module):
    """速度场 u_θ(x, t)：输入 2D 位置 x 与时间 t，输出 2D 速度"""

    def __init__(self, hidden=128, depth=2):
        super().__init__()
        layers, d = [], 3                       # 输入 = 2 个坐标 + 1 个时间
        for _ in range(depth):
            layers += [nn.Linear(d, hidden), nn.SiLU()]
            d = hidden
        layers.append(nn.Linear(d, 2))
        self.net = nn.Sequential(*layers)

    def forward(self, x, t):
        # t 可以是一个数（采样时）或 (B, 1) 的张量（训练时）：统一成 (B, 1)，再拼到 x 后面
        t = torch.as_tensor(t, dtype=x.dtype).reshape(-1, 1).expand(len(x), 1)
        return self.net(torch.cat([x, t], dim=1))


def fm_loss(model, x1):
    # MIT 讲义 Algorithm 3 的一步：抽噪声与时间 → 插值 → 回归直线速度
    x0 = torch.randn_like(x1)
    t = torch.rand(len(x1), 1)
    xt = interpolant(x0, x1, t)
    return ((model(xt, t) - fm_target(x0, x1)) ** 2).mean()


ck.fm_loss(fm_loss)

训练循环就是第 8 周的五步核心；唯一的变化是**每一步都重新抽一批数据**（数据生成器要多少给多少），所以没有 epoch 的概念：

In [ ]:
def train_fm(sample_data, steps=3000, batch_size=512, hidden=128, depth=2, lr=1e-3, seed=0):
    experiments.set_seed(seed)
    model = VelocityMLP(hidden, depth)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    for step in range(steps):
        loss = fm_loss(model, sample_data(batch_size))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return model, losses


t0 = time.time()
moons_model, moons_losses = train_fm(DATA["moons"])
print(f"月牙：3000 步用时 {time.time() - t0:.1f} 秒 | 前 100 步平均损失 {np.mean(moons_losses[:100]):.3f}"
      f" → 最后 100 步 {np.mean(moons_losses[-100:]):.3f}")

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(np.convolve(moons_losses, np.ones(50) / 50, mode="valid"))
ax.set_xlabel("Training step (moving average of 50)")
ax.set_ylabel("FM loss (MSE)")
ax.set_title("The loss plateaus far above zero, and that is expected")
plt.show()

**损失降不到 0，这是正常的**：同一个输入 $(x_t, t)$ 可能来自很多条方向不同的直线，标签本身就互相矛盾——网络最多只能学到它们的平均值，剩下的是不可约的噪声（就像第 4 周给同一个 $x$ 配了不同的 $y$）。所以**损失的大小说明不了样本好不好**，必须采样出来评估。

### 怎么给 2D 样本打分

要比较"生成的点云"和"真实点云"。工具包提供两个指标（属于"课程不教推导、但要会读"的部分）：

- **能量距离**（energy distance；Székely & Rizzo 2013）：$\mathcal{E}(X, Y) = 2\,\mathbb{E}\|X - Y\| - \mathbb{E}\|X - X'\| - \mathbb{E}\|Y - Y'\|$。两个分布相同时为 0，差得越远越大。工具包用无偏估计，所以同分布时它在 0 附近波动、可以**小幅为负**。它对整体位置、尺度、各模式的比例敏感。
- **k-NN precision**（Kynkäänniemi et al. 2019）：以每个真实点为球心、它到第 5 近的真实邻居的距离为半径画球；生成点落在这些球里的比例。（原文默认 $k = 3$，在 VGG-16 的特征空间里、每组 5 万个样本上算；我们直接在 2D 坐标上算、每组 2000 个点，取 $k = 5$——点这么少时 $k = 3$ 的球太细，连"真 vs 真"也只有约 0.97，$k = 5$ 让它接近 1。）它回答"生成的点有多少画在了数据该在的地方"（1 = 没有画到空白处），取值 0 到 1。

任何指标都要先量**噪声地板**：两组独立的真实样本之间，指标本身会波动多少——比这个波动还小的差异不可解读。

In [ ]:
for name, sample in DATA.items():
    eds = [generative.energy_distance(sample(2000), sample(2000)) for _ in range(10)]
    precs = [generative.knn_precision(sample(2000), sample(2000)) for _ in range(10)]
    print(f"{name:17s} 真 vs 真（10 次）：能量距离 {np.mean(eds):+.4f} ± {np.std(eds):.4f} | k-NN precision {np.mean(precs):.3f}")

（工具包的月牙是**独立同分布**采样：每个点独立地随机选弧、随机取角度。`sklearn.datasets.make_moons` 沿弧线**等距**放点，两次调用的样本不是独立同分布，"真 vs 真"的能量距离会稳定地偏到 −0.004——拿它当参考，所有结果都会被系统性地低估。）

现在采样：从 $t = 0$ 的噪声出发，沿学到的速度场用欧拉法走到 $t = 1$。`VelocityMLP` 的调用方式 `model(x, t)` 恰好就是 `euler_integrate` 需要的 `u(x, t)`：

In [ ]:
@torch.no_grad()
def fm_sample(model, n, n_steps=100, seed=0):
    # 从 t = 0 的标准正态噪声出发，用欧拉法沿学到的速度场走到 t = 1；返回整条轨迹
    g = torch.Generator().manual_seed(seed)
    x0 = torch.randn(n, 2, generator=g)
    return euler_integrate(model, x0, n_steps)


torch.manual_seed(123)
moons_ref = DATA["moons"](2000)                       # 固定的真实参考样本
moons_path = fm_sample(moons_model, 2000, 100)
moons_samples = moons_path[-1]
ed = generative.energy_distance(moons_samples, moons_ref)
prec = generative.knn_precision(moons_samples, moons_ref)
print(f"月牙，欧拉 100 步：能量距离 {ed:.4f} | k-NN precision {prec:.3f}")
ck.energy_distance_value(ed)

In [ ]:
def show_samples_and_paths(path, ref, title_left, n_paths=200):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
    axes[0].scatter(ref[:, 0], ref[:, 1], s=2, color="lightgray", label="Real data")
    axes[0].scatter(path[-1, :, 0], path[-1, :, 1], s=2, color="C0", label="Generated")
    axes[0].set_title(title_left)
    axes[0].legend(fontsize=8, markerscale=4)
    for k in range(n_paths):
        axes[1].plot(path[:, k, 0], path[:, k, 1], lw=0.5, color="C0", alpha=0.4)
    axes[1].scatter(path[0, :n_paths, 0], path[0, :n_paths, 1], s=4, color="gray", label="t = 0 (noise)")
    axes[1].scatter(path[-1, :n_paths, 0], path[-1, :n_paths, 1], s=4, color="C3", label="t = 1 (sample)")
    axes[1].set_title("Particle trajectories under the learned velocity field")
    axes[1].legend(fontsize=8)
    for ax in axes:
        ax.set_xlim(-6, 6)
        ax.set_ylim(-4, 4)
        ax.set_aspect("equal")
    plt.tight_layout()
    plt.show()


show_samples_and_paths(moons_path, moons_ref,
                       f"100 Euler steps land on the moons (ED {ed:.3f}, precision {prec:.2f})")

大部分轨迹几乎是直的——训练时用的是直线，学到的平均速度场也接近直线运动（但不完全是：直线会交叉，平均之后轨迹会略微弯曲）。precision 低于噪声地板，主要来自两条弧之间和两端零星的点。

同样的代码换一份数据，不改一行：

In [ ]:
t0 = time.time()
gmm_model, _ = train_fm(DATA["gaussian mixture"])
print(f"双高斯：3000 步用时 {time.time() - t0:.1f} 秒")
torch.manual_seed(123)
gmm_ref = DATA["gaussian mixture"](2000)
gmm_path = fm_sample(gmm_model, 2000, 100)
ed_g = generative.energy_distance(gmm_path[-1], gmm_ref)
prec_g = generative.knn_precision(gmm_path[-1], gmm_ref)
left = float((gmm_path[-1][:, 0] < 0).float().mean())
print(f"双高斯，欧拉 100 步：能量距离 {ed_g:.4f} | k-NN precision {prec_g:.3f} | 落在左边的比例 {left:.3f}（真值 0.5）")
show_samples_and_paths(gmm_path, gmm_ref, f"Two Gaussians (ED {ed_g:.3f}, precision {prec_g:.2f})")

双高斯上，中间的粒子要尽早"决定"去左边还是右边，轨迹在途中分岔。能量距离对**两个模式的比例**很敏感：生成样本里一个模式的占比偏离真值 $\delta$（例如 52% 而不是 50%，$\delta = 0.02$），能量距离就增加 $\delta^2\,\mathcal{E}(A, B)$，其中 $\mathcal{E}(A, B) \approx 9.7$ 是两个高斯分量之间的能量距离（比 $2D = 12$ 略小，$D = 6$ 是两个中心的距离）——所以在双高斯上它的种子间波动会比月牙大。

**小结**：整个 Flow Matching 的训练核心只有 `interpolant`、`fm_target`、`fm_loss` 三个小函数，采样就是欧拉法。下一节看它在什么时候会失败。

---
## S3 · 更难的分布：容量不足与步数不足长什么样

月牙和双高斯太"好学"了。换成棋盘格与双螺旋（改编自 FFJORD 的 toy data，MIT License），对比两个配置：

- **小**：教案的 128×2 层 MLP、3000 步——和 S2 完全一样；
- **大**：256×4 层、10000 步（宽度、深度、训练步数同时加大）。

再加两个对照：用大模型但**只走 5 步**欧拉；以及一个最朴素的"生成器"——与数据同均值、同协方差的**单个高斯**。

In [ ]:
HARD = {"checkerboard": generative.sample_checkerboard, "spiral": generative.sample_spiral}
SMALL = dict(hidden=128, depth=2, steps=3000)
LARGE = dict(hidden=256, depth=4, steps=10000)
hard_models, hard_refs = {}, {}
t0 = time.time()
for dname, sample in HARD.items():
    torch.manual_seed(5)
    hard_refs[dname] = sample(2000)
    hard_models[(dname, "small")] = train_fm(sample, **SMALL)[0]
    hard_models[(dname, "large")] = train_fm(sample, **LARGE)[0]
print(f"4 个模型共用时 {time.time() - t0:.0f} 秒")

In [ ]:
rows = {}
fig, axes = plt.subplots(2, 5, figsize=(17, 7))
for r, (dname, sample) in enumerate(HARD.items()):
    ref = hard_refs[dname]
    torch.manual_seed(6)
    gauss = torch.distributions.MultivariateNormal(ref.mean(0), torch.cov(ref.T)).sample((2000,))
    panels = [("Real data (new draw)", sample(2000)),
              ("One Gaussian, same mean/cov", gauss),
              ("Small model, 100 steps", fm_sample(hard_models[(dname, "small")], 2000, 100)[-1]),
              ("Large model, 5 steps", fm_sample(hard_models[(dname, "large")], 2000, 5)[-1]),
              ("Large model, 100 steps", fm_sample(hard_models[(dname, "large")], 2000, 100)[-1])]
    for col, (title, xs) in enumerate(panels):
        ed = generative.energy_distance(xs, ref)
        prec = generative.knn_precision(xs, ref)
        rows[(dname, title)] = (ed, prec)
        ax = axes[r, col]
        ax.scatter(xs[:, 0], xs[:, 1], s=1, color="gray" if col == 0 else "C0")
        ax.set_title(f"{dname}: {title}\nED {ed:.4f} | precision {prec:.2f}", fontsize=9)
        ax.set_xlim(-4.5, 4.5)
        ax.set_ylim(-4.5, 4.5)
        ax.set_aspect("equal")
fig.suptitle("Energy distance barely separates a blurry blob from good samples; k-NN precision does")
plt.tight_layout()
plt.show()
for (dname, title), (ed, prec) in rows.items():
    print(f"{dname:12s} | {title:28s} | 能量距离 {ed:+.4f} | precision {prec:.3f}")
for dname, sample in HARD.items():
    eds = [generative.energy_distance(sample(2000), hard_refs[dname]) for _ in range(10)]
    print(f"{dname:12s} | 噪声地板（10 次真 vs 真）     | 能量距离 {np.mean(eds):+.4f} ± {np.std(eds):.4f}")

两种典型的失败形态：

- **容量 / 训练不足**（小模型）：一团模糊的"斑块"——它盖住了数据所在的区域，却也把大量的点撒进格子之间、螺旋臂之间的空白处；
- **采样步数不足**（大模型只走 5 步）：粒子还没走到位，结构被"抹开"。

再看指标：小模型的斑块与大模型的好样本，**能量距离只差千分之几**，和噪声地板同一个量级（表格最后两行：连"真 vs 真"本身都会波动 ±0.003 左右，单独一次的"真 vs 真"甚至可能比好模型还大；在螺旋上它甚至可能把两者排反）；而 precision 把它们清楚地分开（约 0.7 vs 0.95）。能量距离衡量的是整体的位置、尺度和比例，对"格子画没画清楚"这种细结构不敏感——连"单个高斯"都只比斑块差一点。

**教训：一个指标的数值好，不等于样本好。**报告两个互补的指标，先量噪声地板，并且一定要把样本画出来亲眼看。

### 从 2D 到图像：U-Net 只是一个更大的 $u_\theta(x, t)$

作业里你会在 MNIST 上做同样的事。图像上的速度场网络用 **U-Net**——它属于"课程不教但绕不开"的组件，由工具包提供 `generative.SmallUNet`（改编自 MIT 6.S184 2025 版 Lab 3 的 `MNISTUNet`，MIT License）。你只需要懂它的**输入与输出**：

In [ ]:
unet = generative.SmallUNet(img_size=14)
x_img = torch.randn(8, 1, 14, 14)              # 8 张 14×14 的单通道"图"
t_img = torch.rand(8)                          # 每张图一个时间
print("参数量:", generative.count_parameters(unet))
print("输入 x", tuple(x_img.shape), "+ 时间 t", tuple(t_img.shape), "→ 输出", tuple(unet(x_img, t_img).shape))

和 `VelocityMLP` 一样：输入 $(x, t)$，输出与 $x$ 同形状的速度。训练代码几乎不用改——只是时间的形状要变。如果照搬 2D 的写法、每张图抽一个数组成 `(B,)` 的时间，会发生什么？**先猜，再运行，读报错的最后一行**：

In [ ]:
# 故意出错：时间抽成 (B,)，而图像是 (B, 1, 14, 14)
x1_img = torch.randn(8, 1, 14, 14)
x0_img = torch.randn_like(x1_img)
t_wrong = torch.rand(8)
xt_img = (1 - t_wrong) * x0_img + t_wrong * x1_img

报错说在第 3 维上 8 和 14 对不上：广播从**最后一维**开始对齐，形状 `(8,)` 被当成了图像的"宽度"。把时间抽成 `(B, 1, 1, 1)`——`torch.rand(8, 1, 1, 1)`——每张图的那一个数就会广播到它自己的所有像素上。

### Capstone：进入执行期

从本周起 Capstone 进入执行期（第 16–20 周）。本周的里程碑是**基线**：一条从原始数据到评估指标、能完整跑通的流水线，加上至少一个基线的结果（本周 1 个种子即可；第 18 周的主结果要求 ≥ 3 个种子）。今天 S2–S3 的做法就是一份基线检查的样板：先量噪声地板（或随机猜、多数类这类平凡基线），再报告互补的指标，最后把结果画出来亲眼看。S4 每人带着自己的基线来。

---
## S4 · Capstone 基线自查

按下面的清单把你的基线写成一页（想练表达的话，再录一段 5–8 分钟的讲解），然后逐项过一遍排错清单。

**基线内容清单**

1. **研究问题与假设**：一句话，与第 12 周设计文档一致；如有改动，说明理由。
2. **数据流水线**：来源、划分方式（训练 / 验证 / 测试各多少，如何避免泄漏）；展示 3–5 个样本及其标签，证明加载正确。
3. **基线**：是什么、为什么选它；主指标是什么、为什么。
4. **基线结果**：数值，并与一个**平凡参照**放在一起——随机猜 / 多数类 / "真 vs 真"的噪声地板；至少一张图。
5. **运行成本**：单次运行多久（Colab 上）；按此估算主实验（≥ 3 个种子 × 所有条件）总共要多少时间。
6. **下一步与最大风险**：第 18 周主结果之前最可能出问题的一件事，以及最便宜的验证办法（第 12 周的 vectoring）。

**排错清单**（第 9 周诊断配方的 Capstone 版）

- [ ] 随机抽几个样本，打印 / 画出输入与标签，确认它们对得上
- [ ] 训练、验证、测试没有重叠（按 ID 或内容哈希检查一次）
- [ ] 初始损失与理论值一致（分类 ≈ ln 类别数；回归 ≈ 目标的方差）
- [ ] 能把一个很小的 batch 过拟合到损失接近 0
- [ ] 与输入无关的基线（全零输入 / 打乱标签）明显更差
- [ ] 指标实现用一个已知答案的例子验证过（例如完美预测 → 指标满分；两组同分布样本 → 距离 ≈ 0）
- [ ] 种子固定后，同一配置跑两次结果一致
- [ ] notebook 能从头到尾 Restart & Run All
- [ ] 这次运行在实验记录本里有一条记录（作业 C）

### 本周小结

- 生成建模 = 把容易采样的噪声**搬运**成数据；ODE $dx/dt = u(x, t)$ 就是"粒子在速度场里流动"
- 欧拉法 $x_{t+h} = x_t + h\,u(x_t, t)$：步数越多越准；采样步数就是欧拉步数
- Flow Matching：线性路径 $x_t = (1-t)x_0 + t x_1$（**$t = 0$ 噪声，$t = 1$ 数据**），目标 $x_1 - x_0$，损失是 MSE——训练就是一个回归问题；损失降不到 0 是正常的
- 评估生成样本：先量噪声地板；能量距离对细结构不敏感，同时报告 k-NN precision，并画出来看
- 失败形态：容量 / 训练不足 → 模糊斑块；采样步数不足 → 结构被抹开
- 图像上的速度场网络是工具包的 `SmallUNet`：输入 $(x, t)$，输出同形状的速度